# Lab 3: Conversation Memory & Tool Calling — SOLUTIONS
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> You are building a clinical assistant for KHCC nurses. The assistant must remember the conversation context (so nurses can ask follow-up questions) and have access to tools that look up patient vitals and check reference ranges.

### Objective
- Build conversation memory using a messages list
- Implement OpenAI tool calling with clinical tools
- Create a multi-turn clinical assistant

---
### Setup

In [ ]:
!pip install -q openai tiktoken

In [ ]:
import json
from openai import OpenAI

# Paste your OpenAI API key here
OPENAI_API_KEY = ""  # <-- paste your key between the quotes

client = OpenAI(api_key=OPENAI_API_KEY)
print("Client ready!")

---
## Cell 1 — The Stateless Problem
First, let's see why memory matters. Make two API calls without memory.

In [ ]:
# === CELL 1: THE STATELESS PROBLEM ===

# Call 1: Ask about a patient
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"}]
)
print("Response 1:", response1.choices[0].message.content)

# Call 2: Follow-up WITHOUT memory
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What should the nurse do about it?"}]
)
print("\nResponse 2:", response2.choices[0].message.content)
print("\n--- Notice: The model has NO idea what 'it' refers to! ---")

---
## Cell 2 — Building Conversation Memory
Maintain a messages list and send the full history each time.

In [ ]:
# === CELL 2: CONVERSATION MEMORY ===

# SOLUTION: Create a messages list with a system message
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions."}
]

# SOLUTION: Write a function that maintains conversation history
def send_message(user_input):
    """Send a message and maintain conversation memory."""
    # 1. Append the user message to the messages list
    messages.append({"role": "user", "content": user_input})

    # 2. Call the API with the full messages list
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    # 3. Extract the assistant reply
    assistant_reply = response.choices[0].message.content

    # 4. Append the assistant response to the messages list
    messages.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply

# Test with a multi-turn conversation
print("Turn 1:")
print(send_message("Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"))

print("\n" + "="*60)
print("Turn 2:")
print(send_message("What should the nurse do about it?"))

print("\n" + "="*60)
print("Turn 3:")
print(send_message("Should we also check their blood pressure?"))

print("\n--- Notice: Turn 2 and 3 correctly reference previous context! ---")
print(f"\nConversation history length: {len(messages)} messages")

---
## Cell 3 — Token Budget Management
As conversations grow, the token count increases. Let's track it.

In [ ]:
# === CELL 3: TOKEN BUDGET ===
import tiktoken

# SOLUTION: Count tokens in the conversation history
def count_conversation_tokens(messages):
    """Count the total tokens in the conversation history."""
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    total_tokens = 0
    for message in messages:
        # Each message has overhead tokens for role, etc.
        total_tokens += 4  # approx overhead per message
        for key, value in message.items():
            if value is not None:
                total_tokens += len(encoding.encode(str(value)))
    total_tokens += 2  # priming tokens
    return total_tokens

# Show token count for our conversation so far
token_count = count_conversation_tokens(messages)
print(f"Current conversation tokens: {token_count}")
print(f"Percentage of 128K context window: {token_count / 128000 * 100:.2f}%")
print(f"\nDiscussion: When the conversation approaches 128K tokens,")
print(f"you need a strategy: truncate old messages, summarize, or start fresh.")

---
## Cell 4 — Clinical Tool Definitions
Define tools the model can call to look up patient data.

In [ ]:
# === CELL 4: CLINICAL TOOLS ===

# Mock patient database (simulating KHCC data)
patient_db = {
    "18887304731609": {
        "vitals": {"temperature": 101.5, "pulse": 95, "bp": "130/85", "spo2": 96, "respiration": 20},
        "ward": "KHCC-4C-HUSSAM AL-HARIRI",
        "last_updated": "2026-01-16T12:32:00"
    },
    "21307935080461": {
        "vitals": {"temperature": 98.6, "pulse": 72, "bp": "120/80", "spo2": 98, "respiration": 16},
        "ward": "KHCC-3B-ONCOLOGY",
        "last_updated": "2026-01-16T14:00:00"
    }
}

# Clinical reference ranges
reference_ranges = {
    "temperature": {"low": 97.8, "high": 99.1, "unit": "°F"},
    "pulse": {"low": 60, "high": 100, "unit": "bpm"},
    "spo2": {"low": 95, "high": 100, "unit": "%"},
    "respiration": {"low": 12, "high": 20, "unit": "breaths/min"}
}

# SOLUTION: Three clinical tool functions

def lookup_vitals(mrn):
    """Look up the latest vitals for a patient by MRN."""
    if mrn in patient_db:
        return {"mrn": mrn, "vitals": patient_db[mrn]["vitals"]}
    return {"error": f"Patient with MRN {mrn} not found"}

def check_abnormal_range(vital_type, value):
    """Check if a vital sign value is within normal clinical range."""
    if vital_type not in reference_ranges:
        return {"error": f"Unknown vital type: {vital_type}"}
    ref = reference_ranges[vital_type]
    if value < ref["low"]:
        status = "LOW"
    elif value > ref["high"]:
        status = "HIGH"
    else:
        status = "normal"
    return {
        "vital": vital_type,
        "value": value,
        "status": status,
        "reference": f"{ref['low']}-{ref['high']} {ref['unit']}"
    }

def get_patient_summary(mrn):
    """Get a full summary of a patient including vitals, ward, and last update."""
    if mrn in patient_db:
        patient = patient_db[mrn]
        return {
            "mrn": mrn,
            "ward": patient["ward"],
            "vitals": patient["vitals"],
            "last_updated": patient["last_updated"]
        }
    return {"error": f"Patient with MRN {mrn} not found"}

# Quick test
print("lookup_vitals test:", lookup_vitals("18887304731609"))
print("check_abnormal_range test:", check_abnormal_range("temperature", 101.5))
print("get_patient_summary test:", get_patient_summary("21307935080461"))

---
## Cell 5 — OpenAI Tool Definitions
Define the tools in OpenAI's JSON Schema format so the model knows what it can call.

In [ ]:
# === CELL 5: OPENAI TOOL DEFINITIONS ===

# SOLUTION: Define tools in OpenAI format
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_vitals",
            "description": "Look up the latest vitals for a patient by their Medical Record Number (MRN).",
            "parameters": {
                "type": "object",
                "properties": {
                    "mrn": {
                        "type": "string",
                        "description": "The patient's Medical Record Number (MRN)"
                    }
                },
                "required": ["mrn"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_abnormal_range",
            "description": "Check if a vital sign value is within normal clinical range. Returns whether the value is normal, high, or low.",
            "parameters": {
                "type": "object",
                "properties": {
                    "vital_type": {
                        "type": "string",
                        "enum": ["temperature", "pulse", "spo2", "respiration"],
                        "description": "The type of vital sign to check"
                    },
                    "value": {
                        "type": "number",
                        "description": "The vital sign value to check"
                    }
                },
                "required": ["vital_type", "value"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_patient_summary",
            "description": "Get a full summary of a patient including vitals, ward location, and last update time.",
            "parameters": {
                "type": "object",
                "properties": {
                    "mrn": {
                        "type": "string",
                        "description": "The patient's Medical Record Number (MRN)"
                    }
                },
                "required": ["mrn"]
            }
        }
    }
]

print(f"Defined {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool['function']['name']}: {tool['function']['description'][:60]}...")

---
## Cell 6 — The Complete Tool Calling Loop
Put it all together: user asks -> model calls tools -> you execute -> model answers.

In [ ]:
# === CELL 6: TOOL CALLING LOOP ===

# Map function names to actual functions
available_functions = {
    "lookup_vitals": lookup_vitals,
    "check_abnormal_range": check_abnormal_range,
    "get_patient_summary": get_patient_summary,
}

# Reset messages for tool-calling conversation
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions. Use the available tools to look up patient data and check vital sign ranges."}
]

# SOLUTION: Complete tool calling loop
def clinical_assistant(user_input):
    """Process user input with tool calling and memory."""
    # 1. Add user message to messages list
    messages.append({"role": "user", "content": user_input})
    print(f"\n{'='*60}")
    print(f"User: {user_input}")

    # 2. Call API with tools
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    # 3. Get the response message
    response_message = response.choices[0].message

    # 4. Append the assistant message to messages (IMPORTANT: must include tool_calls)
    messages.append(response_message)

    # 5. Check if the model wants to call tools
    if response_message.tool_calls:
        print(f"\n--- Model requested {len(response_message.tool_calls)} tool call(s) ---")

        # 6. Execute each tool call
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            print(f"  Calling: {function_name}({function_args})")

            # Call the actual function
            function_to_call = available_functions[function_name]
            result = function_to_call(**function_args)

            print(f"  Result: {result}")

            # Add tool result to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        # 7. Call API again with tool results so model can generate final answer
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        final_message = second_response.choices[0].message
        messages.append(final_message)
        print(f"\nAssistant: {final_message.content}")
        return final_message.content

    else:
        # No tool calls — model responded directly
        print(f"\nAssistant: {response_message.content}")
        return response_message.content

# Test with clinical questions
clinical_assistant("What are the latest vitals for patient 18887304731609?")

In [ ]:
# Additional test: check if temperature is normal
clinical_assistant("Is the patient's temperature normal?")

In [ ]:
# Additional test: get full summary of another patient
clinical_assistant("Give me a full summary of patient 21307935080461")

---
## Cell 7 — Multi-Turn with Tools and Memory
Combine memory with tool calling for a full clinical conversation.

In [ ]:
# === CELL 7: MULTI-TURN WITH TOOLS AND MEMORY ===

# Reset for a clean multi-turn demo
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions. Use the available tools to look up patient data and check vital sign ranges."}
]

# Turn 1: Look up vitals
clinical_assistant("What are the vitals for patient 18887304731609?")

# Turn 2: Follow-up — model uses context from Turn 1 and calls check_abnormal_range
clinical_assistant("Is anything abnormal?")

# Turn 3: Compare with another patient — model looks up second patient and compares
clinical_assistant("Compare with patient 21307935080461")

print(f"\n\n{'='*60}")
print(f"Total messages in conversation: {len(messages)}")
print(f"Total tokens: {count_conversation_tokens(messages)}")

---
## Stretch Challenge
Add a fourth tool: `recommend_action(vital_type, severity)` that suggests clinical actions.

In [ ]:
# === STRETCH: RECOMMEND_ACTION TOOL ===

# SOLUTION: Clinical action recommendation tool
action_recommendations = {
    "temperature": {
        "mild": "Monitor every 2 hours, administer antipyretic (paracetamol) if persistent. Ensure adequate hydration.",
        "severe": "Immediate blood cultures, notify attending physician. Administer antipyretic and consider broad-spectrum antibiotics."
    },
    "pulse": {
        "mild": "Recheck in 15 minutes. Document rhythm. Review current medications for rate-affecting drugs.",
        "severe": "Continuous cardiac monitoring. 12-lead ECG stat. Notify attending physician immediately."
    },
    "spo2": {
        "mild": "Reposition patient, encourage deep breathing. Recheck in 15 minutes. Verify probe placement.",
        "severe": "Initiate supplemental O2 (2-4 L/min via nasal cannula). Call rapid response team if no improvement."
    },
    "respiration": {
        "mild": "Elevate head of bed. Assess for pain or anxiety. Recheck in 30 minutes.",
        "severe": "Assess airway patency. Prepare for possible intubation. Call rapid response team."
    }
}

def recommend_action(vital_type, severity):
    """Recommend clinical action based on vital type and severity."""
    if vital_type not in action_recommendations:
        return {"error": f"Unknown vital type: {vital_type}"}
    if severity not in ["mild", "severe"]:
        return {"error": f"Severity must be 'mild' or 'severe', got: {severity}"}
    return {
        "vital_type": vital_type,
        "severity": severity,
        "recommendation": action_recommendations[vital_type][severity]
    }

# Add to available functions
available_functions["recommend_action"] = recommend_action

# Add tool definition
tools.append({
    "type": "function",
    "function": {
        "name": "recommend_action",
        "description": "Recommend a clinical nursing action based on an abnormal vital sign and its severity level.",
        "parameters": {
            "type": "object",
            "properties": {
                "vital_type": {
                    "type": "string",
                    "enum": ["temperature", "pulse", "spo2", "respiration"],
                    "description": "The type of vital sign that is abnormal"
                },
                "severity": {
                    "type": "string",
                    "enum": ["mild", "severe"],
                    "description": "The severity level of the abnormality"
                }
            },
            "required": ["vital_type", "severity"]
        }
    }
})

# Test the stretch tool
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions. Use the available tools to look up patient data, check vital sign ranges, and recommend actions."}
]

clinical_assistant("Patient 18887304731609 has a high temperature. What should the nurse do?")

---
### KHCC Connection
This tool calling pattern is the foundation for the clinical AI agents you'll build with LangChain in Session 5. The key concepts:

1. **Conversation Memory** — Maintaining a messages list enables multi-turn dialogue
2. **Tool Definitions** — JSON Schema descriptions let the model know what functions are available
3. **Tool Calling Loop** — The pattern of: call API -> execute tools -> call API again is the core agent loop
4. **Clinical Safety** — Tools provide structured, auditable access to patient data

In Session 5, LangChain will abstract this loop into reusable agent architectures.